# Anomaly Filtration Analysis

This notebook analyzes the impact of anomaly filtration on Starlink download performance metrics (latency and throughput). It compares statistical measures before and after applying anomaly detection and filtration techniques to November 2025 measurement data.

It generates maps visualizing the absolute differences in various statistics across countries, helping identify where anomaly filtration has the most significant impact on reported performance metrics.

This requires having the enriched November dataset, before applying any anomaly filtration, as well as the filtered latency dataset with 0.75 Percentile method, and 0.75 Isolation Forest filtered throughput dataset. These can be obtained by running our filtration scripts.

These plots are not directly present in the paper, however data presented in the plots is used in Section 4.3, where the model performance is discussed.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import importlib

import map_plotting_utils
importlib.reload(map_plotting_utils)
from map_plotting_utils import (
    Metric,
    plot_filtering_percentage,
    compute_metric_difference,
    plot_metric_comparison_map,
    plot_before_after_maps
)

In [2]:
df_unfiltered = pd.read_csv('data/download_11_2025_with_sat_density_enriched.csv')
df_filtered_latency = pd.read_csv('data/filtered_download_latency_ms_download_11_2025_with_sat_density_enriched.csv')
df_filtered_throughput = pd.read_csv('data/filtered_download_throughput_mbps_download_11_2025_with_sat_density_enriched.csv')

## Download Latency: Filtered Percentage

In [3]:
df_deleted_latency = df_unfiltered[~df_unfiltered['uuid'].isin(df_filtered_latency['uuid'])].copy()
plot_filtering_percentage(df_unfiltered, df_deleted_latency)

## Download Throughput: Filtered Percentage

In [4]:
df_deleted_throughput = df_unfiltered[~df_unfiltered['uuid'].isin(df_filtered_throughput['uuid'])].copy()
plot_filtering_percentage(df_unfiltered, df_deleted_throughput)

## Download Latency: Abs. Diff. MEDIAN

In [5]:
diff_latency_median = compute_metric_difference(
    df_before=df_unfiltered,
    df_after=df_filtered_latency,
    measurement_column='download_latency_ms',
    metric=Metric.MEDIAN,
    min_measurements=50
)

fig = plot_metric_comparison_map(
    diff_df=diff_latency_median,
    measurement_name='Download Latency (ms)',
)
fig.show()

## Download Throughput: Abs. Diff. MEDIAN

In [6]:
diff_throughput_median = compute_metric_difference(
    df_before=df_unfiltered,
    df_after=df_filtered_throughput,
    measurement_column='download_throughput_mbps',
    metric=Metric.MEDIAN,
    min_measurements=50
)

fig = plot_metric_comparison_map(
    diff_df=diff_throughput_median,
    measurement_name='Download Throughput (Mbps)',
)
fig.show()

## Download Latency: MEDIAN

In [7]:
fig = plot_before_after_maps(
    diff_df=diff_latency_median,
    measurement_name='Download Latency (ms)',
    metric=Metric.MEDIAN
)
fig.show()

## Download Throughput: MEDIAN

In [8]:
fig = plot_before_after_maps(
    diff_df=diff_throughput_median,
    measurement_name='Download Throughput (Mbps)',
    metric=Metric.MEDIAN
)
fig.show()

## Download Latency: STD

In [9]:
diff_latency_std = compute_metric_difference(
    df_before=df_unfiltered,
    df_after=df_filtered_latency,
    measurement_column='download_latency_ms',
    metric=Metric.STD,
    min_measurements=50
)

fig = plot_before_after_maps(
    diff_df=diff_latency_std,
    measurement_name='Download Latency (ms)',
    metric=Metric.STD
)
fig.show()

## Download Throughput - STD

In [10]:
diff_throughput_std = compute_metric_difference(
    df_before=df_unfiltered,
    df_after=df_filtered_throughput,
    measurement_column='download_throughput_mbps',
    metric=Metric.STD,
    min_measurements=50
)

fig = plot_before_after_maps(
    diff_df=diff_throughput_std,
    measurement_name='Download Throughput (Mbps)',
    metric=Metric.STD
)
fig.show()